In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random
import json

# Для воспроизводимости
np.random.seed(42)
random.seed(42)

In [ ]:
# ---------------------------
# 1. Параметры генерации
# ---------------------------
end_date = datetime(2026, 3, 31)
start_date = end_date - timedelta(days=90)

n_orders = 150000
n_users = 50000

# Вертикали с реалистичными ценами для РФ
verticals_map = {
    'AVS': 'Авиабилеты TPO',
    'FUEL': 'Топливо',
    'HOTEL': 'Отели',
    'GROCERY': 'Продукты',
    'NON_FMCG': 'Non-FMCG',
    'RESTAURANT': 'Рестораны'
}
source_systems = list(verticals_map.keys())
verticals_probs = [0.15, 0.2, 0.15, 0.25, 0.15, 0.1]

# Платформы (российские реалии)
platforms = ['Мобильная', 'Мобильный банк', 'Web']
platforms_probs = [0.35, 0.45, 0.2]  # 35% мобильная версия, 45% мобильный банк, 20% веб

# Типы подписки
subscription_types = ['unknown', 'Pro', 'Premium']
subscription_probs = [0.6, 0.25, 0.15]  # 60% unknown, 25% Pro, 15% Premium

In [ ]:
# ---------------------------
# 2. Генерация RAW датасета (таблица с операциями)
# ---------------------------
print("=== Генерация RAW датасета ===")
print("Генерация пользователей...")

# 2.1 Генерируем пользователей
users = []
for user_id in range(1, n_users + 1):
    first_order_date = np.random.choice(pd.date_range(start=start_date, end=end_date, freq='D'))
    first_order_dttm = pd.Timestamp(first_order_date) + pd.Timedelta(
        hours=np.random.randint(0, 23),
        minutes=np.random.randint(0, 59),
        seconds=np.random.randint(0, 59)
    )

    users.append({
        'party_id': user_id,
        'first_order_dttm': first_order_dttm
    })

df_users = pd.DataFrame(users)

print(f"Сгенерировано {len(df_users):,} пользователей")
print("Генерация операций...")

# Функция для генерации реалистичных цен в рублях РФ
def generate_realistic_price(vertical):
    if vertical == 'Авиабилеты TPO':
        # Билеты: эконом 5000-15000, бизнес 20000-50000
        if np.random.random() < 0.7:  # 70% эконом
            return np.random.randint(5000, 15000)
        else:  # 30% бизнес
            return np.random.randint(20000, 50000)

    elif vertical == 'Отели':
        # Отели: хостелы 1000-3000, средние 3000-8000, люкс 8000-25000
        rand = np.random.random()
        if rand < 0.3:  # 30% бюджетные
            return np.random.randint(1000, 3000)
        elif rand < 0.7:  # 40% средние
            return np.random.randint(3000, 8000)
        else:  # 30% люкс
            return np.random.randint(8000, 25000)

    elif vertical == 'Топливо':
        # Топливо: 1000-5000 рублей (полный бак)
        return np.random.randint(1000, 5000)

    elif vertical == 'Продукты':
        # Продукты: 500-5000 рублей (мелкие покупки до крупных)
        return np.random.randint(500, 5000)

    elif vertical == 'Non-FMCG':
        # Non-FMCG: электроника, одежда, мебель - широкий диапазон
        rand = np.random.random()
        if rand < 0.4:  # 40% до 5000
            return np.random.randint(1000, 5000)
        elif rand < 0.7:  # 30% 5000-15000
            return np.random.randint(5000, 15000)
        elif rand < 0.9:  # 20% 15000-50000
            return np.random.randint(15000, 50000)
        else:  # 10% дорогие покупки
            return np.random.randint(50000, 150000)

    elif vertical == 'Рестораны':
        # Рестораны: 500-5000 рублей (средний чек)
        return np.random.randint(500, 5000)

    else:
        return np.random.randint(500, 10000)

# 2.2 Генерируем операции
operations = []
for i in range(n_orders):
    if i % 10000 == 0:
        print(f"Прогресс: {i}/{n_orders}")

    # Выбираем случайного пользователя
    user = df_users.sample(1).iloc[0]
    party_id = user['party_id']
    first_order_dttm = user['first_order_dttm']

    # Дата операции не раньше первой покупки пользователя
    days_range = (end_date - first_order_dttm).days
    if days_range <= 0:
        days_range = 1

    days_after_first = np.random.randint(0, max(1, days_range))
    operation_dttm = first_order_dttm + timedelta(days=days_after_first)

    # Добавляем случайное время
    if days_after_first == 0:
        min_seconds = first_order_dttm.hour * 3600 + first_order_dttm.minute * 60 + first_order_dttm.second
        max_seconds = 23 * 3600 + 59 * 60 + 59
        random_seconds = np.random.randint(min_seconds, max_seconds) if max_seconds > min_seconds else min_seconds
    else:
        random_seconds = np.random.randint(0, 24*3600)

    operation_dttm = operation_dttm.replace(
        hour=random_seconds // 3600,
        minute=(random_seconds % 3600) // 60,
        second=random_seconds % 60,
        microsecond=0
    )

    # Source system
    source_system_cd = np.random.choice(source_systems, p=verticals_probs)
    vertical = verticals_map[source_system_cd]

    # Program product (платформа)
    program_product_desc = np.random.choice(platforms, p=platforms_probs)

    # Nominal price (реалистичная цена)
    nominal_price_rub_amt = generate_realistic_price(vertical)

    # Bundle name (подписка)
    bundle_nm = np.random.choice(subscription_types, p=subscription_probs)

    # High category loyalty accrual (кешбек в повышенной категории)
    # Сумма кешбека: 5-15% от стоимости
    cashback_percent = 0
    if bundle_nm == 'Premium' and vertical in ['Топливо', 'Авиабилеты TPO']:
        cashback_percent = np.random.choice([0, 0.10], p=[0.3, 0.7])  # 10% кешбек
    elif bundle_nm == 'Pro':
        cashback_percent = np.random.choice([0, 0.05], p=[0.5, 0.5])   # 5% кешбек
    elif bundle_nm == 'unknown':
        cashback_percent = np.random.choice([0, 0.03], p=[0.8, 0.2])   # 3% кешбек

    if cashback_percent > 0:
        high_category_loyalty_accrual_rub_amt = int(nominal_price_rub_amt * cashback_percent)
    else:
        high_category_loyalty_accrual_rub_amt = 0

    # Bank offer loyalty accrual (скидка/промокод от банка)
    week_number = operation_dttm.isocalendar()[1]
    base_offer_prob = 0.25 + 0.05 * (week_number % 4)
    if vertical == 'Non-FMCG':
        base_offer_prob += 0.15
    if bundle_nm in ['Pro', 'Premium']:
        base_offer_prob += 0.1

    has_bank_offer = np.random.random() < base_offer_prob
    if has_bank_offer:
        # Скидка 5-20% от стоимости
        discount_percent = np.random.uniform(0.05, 0.20)
        bank_offer_loyalty_accrual_rub_amt = int(nominal_price_rub_amt * discount_percent)
    else:
        bank_offer_loyalty_accrual_rub_amt = 0

    # Payment fields (определяем тип оплаты)
    # Вероятность целевого платежа (кредитка или рассрочка)
    target_prob = 0.35  # базовая доля

    if bank_offer_loyalty_accrual_rub_amt > 0:
        target_prob += 0.20
    if bundle_nm in ['Pro', 'Premium']:
        target_prob += 0.10
    if vertical in ['Авиабилеты TPO', 'Отели', 'Non-FMCG']:
        target_prob += 0.10
    target_prob = min(0.92, target_prob)

    # Доля рассрочки среди целевых платежей
    if vertical in ['Авиабилеты TPO', 'Отели', 'Non-FMCG']:
        bnpl_share = 0.55
    elif vertical == 'Топливо':
        bnpl_share = 0.20
    else:
        bnpl_share = 0.35

    # Чем дороже товар, тем чаще рассрочка
    if nominal_price_rub_amt > 30000:
        bnpl_share += 0.25
    elif nominal_price_rub_amt > 15000:
        bnpl_share += 0.15
    elif nominal_price_rub_amt > 5000:
        bnpl_share += 0.05
    bnpl_share = min(0.85, bnpl_share)

    rand = np.random.random()
    if rand < target_prob:
        if np.random.random() < bnpl_share:
            # Рассрочка (BNPL)
            payment_method_cd = 'BNP'
            pos_application_rk = None
            bnpl_order_rk = np.random.randint(100000, 999999)
            financial_account_type_ = None
        else:
            # Кредитная карта
            payment_method_cd = 'CC'
            pos_application_rk = None
            bnpl_order_rk = None
            financial_account_type_ = 'CREDIT_CARD'
    else:
        # Нецелевые платежи
        if np.random.random() < 0.6:
            # POS-терминал (дебетовая карта)
            payment_method_cd = 'POS'
            pos_application_rk = np.random.randint(100000, 999999)
            bnpl_order_rk = None
            financial_account_type_ = 'DEBIT_CARD'
        else:
            # Другие способы
            payment_method_cd = 'OTHER'
            pos_application_rk = None
            bnpl_order_rk = None
            financial_account_type_ = np.random.choice(['DEBIT_CARD', 'ELECTRONIC_WALLET', 'CASH'])

    # Order status (85% успешных, 15% неуспешных)
    status_rand = np.random.random()
    if status_rand < 0.85:
        order_status = 'SUC'
    elif status_rand < 0.92:
        order_status = 'CAN'
    elif status_rand < 0.97:
        order_status = 'ERR'
    else:
        order_status = 'REF'

    operations.append([
        operation_dttm, party_id, source_system_cd, program_product_desc,
        nominal_price_rub_amt, bundle_nm, high_category_loyalty_accrual_rub_amt,
        bank_offer_loyalty_accrual_rub_amt, payment_method_cd, pos_application_rk,
        bnpl_order_rk, financial_account_type_, order_status, first_order_dttm
    ])

# Создаем RAW датасет
raw_cols = [
    'created_dttm', 'party_id', 'source_system_cd', 'program_product_desc',
    'nominal_price_rub_amt', 'bundle_nm', 'high_category_loyalty_accrual_rub_amt',
    'bank_offer_loyalty_accrual_rub_amt', 'payment_method_cd', 'pos_application_rk',
    'bnpl_order_rk', 'financial_account_type_', 'order_status', 'first_order_dttm'
]
df_raw = pd.DataFrame(operations, columns=raw_cols)

print(f"\nRAW датасет создан: {len(df_raw):,} записей")
print(f"Средняя стоимость заказа: {df_raw['nominal_price_rub_amt'].mean():.2f} руб.")
print(f"Медианная стоимость заказа: {df_raw['nominal_price_rub_amt'].median():.2f} руб.")

# Сохраняем RAW датасет
df_raw.to_csv('raw_transaction_data.csv', index=False)
print("RAW датасет сохранен в 'raw_transaction_data.csv'")


=== Генерация RAW датасета ===
Генерация пользователей...
Сгенерировано 50,000 пользователей
Генерация операций...
Прогресс: 0/150000
Прогресс: 10000/150000
Прогресс: 20000/150000
Прогресс: 30000/150000
Прогресс: 40000/150000
Прогресс: 50000/150000
Прогресс: 60000/150000
Прогресс: 70000/150000
Прогресс: 80000/150000
Прогресс: 90000/150000
Прогресс: 100000/150000
Прогресс: 110000/150000
Прогресс: 120000/150000
Прогресс: 130000/150000
Прогресс: 140000/150000

RAW датасет создан: 150,000 записей
Средняя стоимость заказа: 8396.09 руб.
Медианная стоимость заказа: 3832.00 руб.
RAW датасет сохранен в 'raw_transaction_data.csv'


In [ ]:
df_raw.head(20)

,created_dttm,party_id,source_system_cd,program_product_desc,nominal_price_rub_amt,bundle_nm,high_category_loyalty_accrual_rub_amt,bank_offer_loyalty_accrual_rub_amt,payment_method_cd,pos_application_rk,bnpl_order_rk,financial_account_type_,order_status,first_order_dttm
0,2026-02-17 12:59:17,26299,GROCERY,Мобильный банк,4796,Premium,0,0,CC,NaN,NaN,CREDIT_CARD,SUC,2026-01-08 20:00:40
1,2026-03-03 09:22:55,27127,HOTEL,Мобильный банк,2067,unknown,62,0,CC,NaN,NaN,CREDIT_CARD,SUC,2026-02-17 03:11:21
2,2026-02-17 07:21:18,16348,GROCERY,Мобильная,3692,Pro,0,0,CC,NaN,NaN,CREDIT_CARD,SUC,2026-02-14 20:23:27
3,2026-03-24 07:04:19,35640,GROCERY,Web,3796,Premium,0,0,POS,803071.0,NaN,DEBIT_CARD,SUC,2026-03-21 03:08:35
4,2026-03-15 00:47:42,1635,AVS,Web,45301,Premium,4530,0,OTHER,NaN,NaN,CASH,SUC,2026-02-21 09:06:01
5,2026-02-27 09:06:29,26142,NON_FMCG,Мобильная,15193,unknown,0,0,POS,962097.0,NaN,DEBIT_CARD,SUC,2026-02-01 16:50:23
6,2026-03-28 04:34:13,35571,NON_FMCG,Мобильный банк,2887,Pro,0,0,OTHER,NaN,NaN,ELECTRONIC_WALLET,SUC,2026-02-16 11:30:44
7,2026-02-28 11:44:07,14369,FUEL,Мобильный банк,4779,unknown,0,0,POS,905924.0,NaN,DEBIT_CARD,SUC,2026-02-13 12:05:34
8,2026-03-28 02:20:55,44306,HOTEL,Web,1509,unknown,0,0,CC,NaN,NaN,CREDIT_CARD,CAN,2026-02-25 06:25:45
9,2026-03-10 22:55:52,29055,FUEL,Web,2817,unknown,0,0,POS,538503.0,NaN,DEBIT_CARD,CAN,2026-02-05 04:58:23
